[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/04_neural_networks/04_neural_networks_solutions.ipynb)

# 04. Neural Networks — 연습 문제 해설

[04_neural_networks.ipynb](04_neural_networks.ipynb) 끝의 연습 문제 3개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision matplotlib

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

## 연습 1. XOR — 은닉 유닛 수를 8 -> 2로 줄이면?

In [ ]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]], device=device)
Y = torch.tensor([[0.], [1.], [1.], [0.]], device=device)

def make_mlp(hidden_size):
    return nn.Sequential(
        nn.Linear(2, hidden_size),
        nn.Sigmoid(),
        nn.Linear(hidden_size, 1),
        nn.Sigmoid(),
    ).to(device)

def train(model, epochs=3000, lr=0.5):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    losses = []
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(0)
model_h2 = make_mlp(2)
losses_h2 = train(model_h2)

torch.manual_seed(0)
model_h8 = make_mlp(8)
losses_h8 = train(model_h8)

with torch.no_grad():
    print("hidden=2 예측:", model_h2(X).cpu().numpy().round(3).flatten(), " 최종 loss:", losses_h2[-1])
    print("hidden=8 예측:", model_h8(X).cpu().numpy().round(3).flatten(), " 최종 loss:", losses_h8[-1])

In [ ]:
plt.plot(losses_h2, label="hidden=2")
plt.plot(losses_h8, label="hidden=8")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("은닉 유닛 수에 따른 XOR 학습")
plt.legend()
plt.show()

**해설**
- 이론적으로는 은닉 유닛 **2개**만 있어도 XOR을 표현할 수 있습니다 (XOR = 두 개의 직선 결정 경계 조합으로 구성 가능).
- 하지만 실제로 학습해보면 `hidden=2`는 초기 가중치(random seed)에 따라 **local minimum에 갇혀서 잘 못 풀 때가 많습니다** — 표현력은 충분해도 그 해를 SGD로 찾아내는 게 어렵습니다.
- `hidden=8`처럼 여유 있게 유닛을 두면(over-parameterize) 최적화 지형(loss landscape)이 더 우호적이 되어 안정적으로 수렴합니다.
- 실무 교훈: "이론적 최소 크기"보다 약간 여유 있는 모델이 오히려 학습이 더 쉬운 경우가 많습니다.

## 연습 2 & 3. MNIST — Dropout 유무 비교, 98%+ 달성하기

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

class MnistMLP(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(28 * 28, 256), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        layers += [nn.Linear(256, 128), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        layers.append(nn.Linear(128, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total

def train_mnist(model, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    train_accs, test_accs = [], []
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        train_acc = evaluate(model, train_loader)
        test_acc = evaluate(model, test_loader)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        print(f"  epoch {epoch+1}/{epochs}  train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")
    return train_accs, test_accs

In [ ]:
EPOCHS = 5

print("[Dropout 없음]")
torch.manual_seed(0)
model_no_dropout = MnistMLP(dropout_p=0.0).to(device)
train_accs_nd, test_accs_nd = train_mnist(model_no_dropout, EPOCHS)

print("\n[Dropout=0.3]")
torch.manual_seed(0)
model_dropout = MnistMLP(dropout_p=0.3).to(device)
train_accs_d, test_accs_d = train_mnist(model_dropout, EPOCHS)

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs_range, train_accs_nd, "--", label="train (no dropout)")
axes[0].plot(epochs_range, test_accs_nd, label="test (no dropout)")
axes[0].set_title("Dropout 없음: train vs test")
axes[0].legend()

axes[1].plot(epochs_range, train_accs_d, "--", label="train (dropout=0.3)")
axes[1].plot(epochs_range, test_accs_d, label="test (dropout=0.3)")
axes[1].set_title("Dropout=0.3: train vs test")
axes[1].legend()
plt.show()

print(f"최종 gap (train-test) — Dropout 없음: {train_accs_nd[-1]-test_accs_nd[-1]:.4f}")
print(f"최종 gap (train-test) — Dropout=0.3 : {train_accs_d[-1]-test_accs_d[-1]:.4f}")

**해설 (연습 2)**
- Dropout 없이 학습하면 train accuracy가 test accuracy보다 눈에 띄게 높아지는 경향(=train-test gap 확대)이 나타납니다. 모델이 학습 데이터의 세부 패턴(노이즈 포함)까지 암기하기 시작했다는 신호입니다.
- Dropout=0.3을 적용하면 매 스텝마다 뉴런의 일부를 무작위로 꺼서 특정 뉴런 조합에 의존하지 못하게 하므로, train accuracy는 약간 낮아지지만 **train-test gap이 줄어듭니다** — 더 일반화가 잘 된다는 뜻입니다.
- MNIST는 비교적 쉬운 데이터셋이라 이 문제 규모에서는 차이가 크지 않을 수 있지만, 더 복잡한 데이터/모델일수록 Dropout의 효과가 뚜렷해집니다.

**해설 (연습 3)**
위 Dropout=0.3 모델을 5 epoch만 학습해도 보통 test accuracy가 97~98%대에 도달합니다. epoch을 8~10으로 늘리거나 은닉층을 하나 더 추가하면 98% 이상을 안정적으로 넘길 수 있습니다 (원본 강의 Lec 10의 목표와 동일). 정확도를 더 올리고 싶다면 CNN(다음 노트북)을 쓰는 것이 가장 효과적입니다.